# S-band Analysis Notebook

**Product:** NISAR S-band L2 GCOV  
**Acquisitions:** 2026-08-19, 2026-09-24, 2026-10-18  
**CRS:** EPSG:32645 | **Nominal resolution:** 10 m

This notebook inspects and documents the final S-band deliverables. It is deliberately lightweight: do not run full-scene processing here while Script 30 is running.


## 1. Deliverable paths

Expected structure:

```text
S_BAND/
├── raw_data/
├── extracted_features/
├── classification/
├── validation/
├── metadata/
└── notebook/
```


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

# Change this to your actual TEAM_DELIVERABLES location.
DELIVERABLE_DIR = Path(r"C:\Users\HP\Downloads\NISAR\TEAM_DELIVERABLES\S_BAND")

RAW_DIR = DELIVERABLE_DIR / "raw_data"
FEATURE_DIR = DELIVERABLE_DIR / "extracted_features"
CLASS_DIR = DELIVERABLE_DIR / "classification"
VALID_DIR = DELIVERABLE_DIR / "validation"
META_DIR = DELIVERABLE_DIR / "metadata"

DATES = ["20260819", "20260924", "20261018"]

for folder in [RAW_DIR, FEATURE_DIR, CLASS_DIR, VALID_DIR, META_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Deliverable directory:", DELIVERABLE_DIR)


## 2. Acquisition metadata

The following records known product-level information. Add exact source filenames and product identifiers from the original HDF5 metadata; do not guess missing fields.


In [ ]:
acquisition_details = {
    "sensor": "NISAR S-band",
    "product_level": "L2 GCOV",
    "acquisitions": DATES,
    "crs": "EPSG:32645",
    "nominal_pixel_spacing_m": 10,
    "scene_width_pixels": 34200,
    "scene_height_pixels": 33120
}

metadata_path = META_DIR / "acquisition_details.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(acquisition_details, f, indent=4)

print("Saved:", metadata_path)


## 3. Feature-layer inspection

Required output: `extracted_features/available_feature_layers.tif`

This cell only inspects the exported feature stack. It does not regenerate it.


In [ ]:
FEATURE_STACK = FEATURE_DIR / "available_feature_layers.tif"

if FEATURE_STACK.exists():
    with rasterio.open(FEATURE_STACK) as src:
        print("CRS:", src.crs)
        print("Dimensions:", src.width, "x", src.height)
        print("Bands:", src.count)
        print("Dtypes:", src.dtypes)
        print("NoData:", src.nodata)
        print("Band descriptions:", src.descriptions)
else:
    print("Not created yet:", FEATURE_STACK)


## 4. Classification output inspection

Required outputs: `water_mask.tif`, `water_probability.tif`, `backscatter_map.png`.

Only call a raster a probability map if its values represent calibrated probabilities. Otherwise document it as a confidence score.


In [ ]:
outputs = {
    "water_mask": CLASS_DIR / "water_mask.tif",
    "water_probability": CLASS_DIR / "water_probability.tif",
    "backscatter_map": CLASS_DIR / "backscatter_map.png"
}

for name, path in outputs.items():
    print(name, ":", "PRESENT" if path.exists() else "NOT CREATED", "|", path)
    if path.suffix.lower() == ".tif" and path.exists():
        with rasterio.open(path) as src:
            print("  CRS:", src.crs, "| shape:", (src.height, src.width),
                  "| dtype:", src.dtypes, "| nodata:", src.nodata)


## 5. Lightweight raster preview

This uses a reduced-resolution read for visual inspection, not quantitative analysis.


In [ ]:
def preview_raster(path, title, max_side=1200):
    path = Path(path)
    if not path.exists():
        print("Missing:", path)
        return
    with rasterio.open(path) as src:
        scale = max(src.width / max_side, src.height / max_side, 1)
        h = max(1, int(src.height / scale))
        w = max(1, int(src.width / scale))
        arr = src.read(1, out_shape=(h, w),
                       resampling=rasterio.enums.Resampling.nearest).astype("float32")
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
    plt.figure(figsize=(9, 7))
    plt.imshow(arr)
    plt.title(title)
    plt.axis("off")
    plt.colorbar(shrink=0.75)
    plt.show()

# Uncomment after the files have been generated:
# preview_raster(CLASS_DIR / "water_mask.tif", "S-band water mask")
# preview_raster(CLASS_DIR / "water_probability.tif", "S-band probability/confidence")


## 6. Validation outputs

Required: `metrics.csv` and `confusion_matrix.csv`.

Use independent reference labels aligned to the same grid. Do not report training-set performance as independent validation.


In [ ]:
for filename in ["metrics.csv", "confusion_matrix.csv"]:
    path = VALID_DIR / filename
    if path.exists():
        print("\\n" + filename)
        display(pd.read_csv(path))
    else:
        print("Not available yet:", path)


## 7. Preprocessing record

Edit this file so it describes only the processing actually performed.


In [ ]:
preprocessing_text = '''
Sensor/product: NISAR S-band L2 GCOV
Acquisitions: 2026-08-19, 2026-09-24, 2026-10-18
CRS: EPSG:32645
Nominal resolution: 10 m

Processing summary:
- Original GCOV products retained in raw_data.
- Product masks and input-data exception flags inspected.
- SSAR-shift flags retained as quality information, not automatically treated as corrupted pixels.
- Gamma0/Sigma0 interpretation documented from original product metadata.
- Feature stack exported to extracted_features.
- Water classification exported to classification.
- Validation performed against independent reference labels, if available.

Update this summary to match the final executed workflow.
'''
path = META_DIR / "preprocessing.txt"
path.write_text(preprocessing_text.strip() + "\\n", encoding="utf-8")
print("Saved:", path)


## 8. Final deliverable checklist

Run this at the end. Missing validation files must remain marked missing if independent reference data is unavailable; do not create placeholder metrics.


In [ ]:
required = [
    RAW_DIR / "original_GCOV_product.h5",
    FEATURE_DIR / "available_feature_layers.tif",
    CLASS_DIR / "water_mask.tif",
    CLASS_DIR / "water_probability.tif",
    CLASS_DIR / "backscatter_map.png",
    VALID_DIR / "metrics.csv",
    VALID_DIR / "confusion_matrix.csv",
    META_DIR / "acquisition_details.json",
    META_DIR / "preprocessing.txt"
]

check = pd.DataFrame({
    "deliverable": [str(x.relative_to(DELIVERABLE_DIR)) for x in required],
    "exists": [x.exists() for x in required]
})
display(check)
print(f"Present: {check['exists'].sum()} / {len(check)}")


## 9. Limitations

Record any missing inputs, invalid-data handling, reference-label source and date, grid-alignment limitations, and whether the probability layer is calibrated. Do not claim independent accuracy if suitable independent reference labels were unavailable.
